# DQ-09 · shipments.csv
Checks: null rate, duplicates, FK, temporal chain (order → ship → delivery), gap flags.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── helpers ───────────────────────────────────────────────────────────────────
issues = []

def flag(label, mask_or_count, df=None, show_sample=True):
    count = int(mask_or_count.sum()) if hasattr(mask_or_count, 'sum') else int(mask_or_count)
    total = len(df) if df is not None else None
    pct   = f' ({count/total*100:.2f}%)' if total else ''
    status = '❌ ISSUE' if count > 0 else '✅ OK'
    print(f'{status}  {label}: {count:,}{pct}')
    if count > 0:
        issues.append(label)
        if show_sample and df is not None and hasattr(mask_or_count, 'sum'):
            sample = df[mask_or_count].head(3)
            print(sample.to_string(index=False))
    return count

def summary():
    print()
    if issues:
        print(f'══ {len(issues)} issue(s) found ══')
        for i in issues: print(f'  • {i}')
    else:
        print('══ All checks passed ══')


In [ ]:
ship   = pd.read_csv('shipments.csv', parse_dates=['ship_date','delivery_date'])
orders = pd.read_csv('orders.csv',    parse_dates=['order_date'])
print(f'Shape: {ship.shape}')
ship.head(3)

## 1. Null rate

In [ ]:
null_counts = ship.isnull().sum()
print(pd.DataFrame({'null_count': null_counts, 'null_%': (null_counts/len(ship)*100).round(2)}))

## 2. 1:0-or-1 với orders (mỗi order tối đa 1 shipment)

In [ ]:
flag('Duplicate order_id in shipments', ship.duplicated(subset='order_id'), ship)

## 3. FK: order_id → orders

In [ ]:
valid_orders = set(orders['order_id'])
flag('order_id not in orders', ~ship['order_id'].isin(valid_orders), ship)

## 4. Chỉ đơn shipped/delivered/returned mới có shipment

In [ ]:
shippable = set(orders[orders['order_status'].isin(['shipped','delivered','returned'])]['order_id'])
flag('Shipment for non-shippable order status', ~ship['order_id'].isin(shippable), ship)

## 5. Temporal chain: order_date ≤ ship_date ≤ delivery_date

In [ ]:
df = ship.merge(orders[['order_id','order_date']], on='order_id', how='left')

flag('ship_date < order_date',    df['ship_date']     < df['order_date'], df)
flag('delivery_date < ship_date', df['delivery_date'] < df['ship_date'],  df)

proc_gap = (df['ship_date']     - df['order_date']).dt.days
del_gap  = (df['delivery_date'] - df['ship_date']).dt.days
tot_gap  = (df['delivery_date'] - df['order_date']).dt.days

print('\nProcessing gap (order → ship) days:');  print(proc_gap.describe().round(1))
print('\nDelivery gap  (ship → delivery) days:'); print(del_gap.describe().round(1))
print('\nTotal gap     (order → delivery) days:'); print(tot_gap.describe().round(1))

## 6. Flag bất thường về gap

In [ ]:
df['proc_gap'] = (df['ship_date']     - df['order_date']).dt.days
df['del_gap']  = (df['delivery_date'] - df['ship_date']).dt.days

flag('Processing gap > 14 days (order→ship)',     df['proc_gap'] > 14, df)
flag('Processing gap < 0 days',                   df['proc_gap'] < 0,  df)
flag('Delivery gap > 30 days (ship→delivery)',     df['del_gap']  > 30, df)
flag('Delivery gap < 0 days',                     df['del_gap']  < 0,  df)

## 7. Business rule: shipping_fee ≥ 0

In [ ]:
flag('shipping_fee < 0', ship['shipping_fee'] < 0, ship)

## Summary

In [ ]:
summary()